# COL-02 — Bounded Baseline Parameter Sweep on Development Pilot

Evaluates the impact of hyperparameters and rerun reproducibility across density clustering baselines on the DAT-02 pilot feature bundle (279 candidate rows, 2 mainshock windows).

### Sweep Constraints:
- Maximum 8 parameter configurations (4 DBSCAN, 4 HDBSCAN_GLOBAL).
- WINDOW baseline as invariant control.
- Strict 4-D normalized vector `[x_scaled, y_scaled, z_scaled, time_scaled]`.
- Non-mutation of input candidate population.
- Isolated output directories per configuration.
- Rerun invariant verification.

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'notebooks/parameter_sweep.py').is_file(), 'Mở notebook từ root repo hoặc notebooks/'
sys.path.insert(0, str(root / 'notebooks'))
sys.path.insert(0, str(root / 'tests'))

from experiment_harness import load_feature_bundle, MODE_DEVELOPMENT
from parameter_sweep import (
    SWEEP_CONFIG_FILES,
    run_bounded_parameter_sweep,
    verify_sweep_rerun_invariants,
)
print('COL-02 Parameter Sweep modules imported successfully.')

## 1. Load Development Pilot Bundle

Loads the DAT-02 pilot bundle. Validates checksums, schema, and non-null constraints before running the sweep.

In [ ]:
from test_experiment_harness import create_pilot_fixture

bundle_path = os.environ.get('PILOT_BUNDLE', '')
_temp_dir = None
if bundle_path:
    pilot_dir = Path(bundle_path).resolve()
    assert pilot_dir.is_dir(), f"PILOT_BUNDLE path does not exist or is not a directory: {bundle_path}"
    input_kind = 'REAL_PILOT'
else:
    _temp_dir = tempfile.TemporaryDirectory()
    pilot_dir = Path(_temp_dir.name) / 'bundle'
    pilot_dir.mkdir(parents=True, exist_ok=True)
    create_pilot_fixture(pilot_dir)
    input_kind = 'SYNTHETIC_FIXTURE'

bundle = load_feature_bundle(pilot_dir, mode=MODE_DEVELOPMENT)
assert bundle['mode'] == MODE_DEVELOPMENT
assert bundle['is_official'] is False

features = bundle['features_table']
print('Loaded Pilot ID:', bundle['bundle_id'])
print('Input kind:', input_kind)
print('Feature candidate count:', features.num_rows)
print('Mainshock count:', bundle['mainshocks_table'].num_rows)
assert features.num_rows > 0


## 2. Execute Bounded Parameter Sweep

Executes all 8 bounded hyperparameter configurations alongside the WINDOW control baseline.
Captures candidate counts, sequence members, noise events, and runtime for each configuration.

In [ ]:
output_env = os.environ.get('COL02_OUTPUT_DIR')
if output_env:
    sweep_out_dir = Path(output_env).resolve()
else:
    sweep_out_dir = (root / 'staging/ml-experiments/col02-sweep').resolve()
allow_overwrite_flag = os.environ.get('ALLOW_OVERWRITE', '0') == '1'

sweep = run_bounded_parameter_sweep(
    bundle=bundle,
    project_root=root,
    output_dir=sweep_out_dir,
    allow_overwrite=allow_overwrite_flag,
    code_version='col02-sweep-v1',
)

matrix = sweep['matrix']
print(f"Sweep completed successfully. Configurations evaluated: {len(matrix['configurations'])}")
print('\nParameter Sweep Results Summary:')
print('-' * 95)
print(f"{'Config':<14} | {'Algorithm':<15} | {'Version':<20} | {'Seq':<6} | {'Noise':<6} | {'Status':<10}")
print('-' * 95)
for r in matrix['configurations']:
    status_str = f"{r['successful_windows']}/{r['windows_processed']} PASS"
    print(f"{r['config_key']:<14} | {r['algorithm_name']:<15} | {r['model_config_version']:<20} | {r['total_sequence_members']:<6} | {r['total_noise']:<6} | {status_str:<10}")
print('-' * 95)


## 3. Verify Rerun Invariance

Reruns a configuration to an isolated output directory to verify deterministic partition invariance and numerical tolerance.

In [ ]:
rerun_dir = sweep_out_dir / 'rerun_check'
rerun_sweep = run_bounded_parameter_sweep(
    bundle=bundle,
    project_root=root,
    output_dir=rerun_dir,
    config_keys=['DBSCAN_C1'],
    allow_overwrite=allow_overwrite_flag,
)

invariants = verify_sweep_rerun_invariants(
    sweep['results']['DBSCAN_C1'],
    rerun_sweep['results']['DBSCAN_C1'],
)
print('Rerun Invariance Check:', invariants['status'])
print(f"Candidates verified: {invariants['total_candidates']} (rtol={invariants['rtol']}, atol={invariants['atol']})")
assert invariants['status'] == 'RERUN_INVARIANTS_PASSED'


## 4. Formal Sweep Report

Saves `col02-sweep-report.json` documenting the parameter sweep matrix, rerun invariants, and DEVELOPMENT_ONLY boundaries.

In [ ]:
from datetime import datetime, timezone

report_path = sweep_out_dir / 'col02-sweep-report.json'
report = {
    'report_version': 'col-02-sweep-report-v1',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_kind': input_kind,
    'is_synthetic_fixture': (input_kind == 'SYNTHETIC_FIXTURE'),
    'pilot_id': bundle['bundle_id'],
    'candidate_count': features.num_rows,
    'mainshock_count': bundle['mainshocks_table'].num_rows,
    'sweep_configurations_evaluated': len(matrix['configurations']),
    'configurations': matrix['configurations'],
    'rerun_verification': invariants,
    'allowed_usage': 'DEVELOPMENT_ONLY',
    'research_ready': False,
    'official_exported': False,
    'scientific_approval': False,
}
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Formal Sweep Report saved to:', report_path)

if _temp_dir is not None:
    _temp_dir.cleanup()
print('Sweep output artifacts preserved at:', sweep_out_dir)
